# Investigación e Implementación de Skills en Gemini
## Cuaderno de Datos (Data Notebook)

**Institución:** Corporación Universitaria Unilasallista  
**Tema:** Investigación e implementación de Skills en Gemini  
**Herramientas:** Python, pandas y matplotlib

> **Nota metodológica:** Los datos de este cuaderno son simulados con fines académicos. No representan mediciones reales de Gemini ni permiten afirmar el rendimiento real de una Skill. Sustituya los datos de ejemplo por resultados obtenidos mediante pruebas controladas antes de formular conclusiones empíricas.


## 1. Objetivo del cuaderno

Diseñar un procedimiento reproducible para evaluar una Skill de Gemini orientada a tareas profesionales, considerando:

- **Tiempo de respuesta:** duración de la ejecución, en segundos.
- **Precisión o tasa de acierto:** proporción de resultados que cumplen criterios definidos previamente.
- **Utilidad operativa:** potencial reducción del tiempo frente a un proceso de referencia.
- **Trazabilidad:** registro de consulta, salida, resultado esperado y evaluación.

La evaluación debe combinar métricas cuantitativas con revisión humana de calidad, pertinencia, seguridad y verificabilidad.


## 2. Estructura de datos de prueba

Cada fila representa una ejecución. `acierto` vale `1` si la respuesta cumple los criterios definidos y `0` si no. `tiempo_skill_s` es el tiempo simulado de la Skill; `tiempo_base_s` es el tiempo de referencia estimado sin la Skill.

| Variable | Tipo | Descripción |
|---|---|---|
| `id_caso` | texto | Identificador único |
| `caso_uso` | texto | Área profesional evaluada |
| `consulta` | texto | Solicitud enviada |
| `salida_resumida` | texto | Síntesis de la respuesta |
| `tiempo_skill_s` | numérico | Tiempo de respuesta, en segundos |
| `tiempo_base_s` | numérico | Tiempo de referencia, en segundos |
| `acierto` | entero binario | 1 = cumple criterios; 0 = no cumple |
| `observaciones` | texto | Comentarios del evaluador |

Los criterios de acierto deben definirse antes de ejecutar las pruebas.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Datos simulados: demostración del flujo de análisis, no mediciones reales.
datos = [
    {"id_caso":"C01","caso_uso":"Gestión empresarial","consulta":"Resume una reunión y extrae tres acciones prioritarias.","salida_resumida":"Resumen y lista de acciones con responsables sugeridos.","tiempo_skill_s":8.4,"tiempo_base_s":420.0,"acierto":1,"observaciones":"Revisar responsables con el equipo."},
    {"id_caso":"C02","caso_uso":"Educación","consulta":"Propón una rúbrica para evaluar un proyecto académico.","salida_resumida":"Rúbrica con criterios, niveles y descriptores.","tiempo_skill_s":10.2,"tiempo_base_s":600.0,"acierto":1,"observaciones":"La estructura cumple el formato solicitado."},
    {"id_caso":"C03","caso_uso":"Análisis de datos","consulta":"Explica cómo resumir una tabla de ventas por región.","salida_resumida":"Pasos de agrupación y cálculo de totales.","tiempo_skill_s":7.9,"tiempo_base_s":300.0,"acierto":1,"observaciones":"Verificar cálculos con datos reales."},
    {"id_caso":"C04","caso_uso":"Comunicación","consulta":"Redacta un borrador de correo para informar un cambio.","salida_resumida":"Correo con asunto, contexto y siguiente paso.","tiempo_skill_s":6.8,"tiempo_base_s":180.0,"acierto":1,"observaciones":"Ajustar el tono antes del envío."},
    {"id_caso":"C05","caso_uso":"Gestión empresarial","consulta":"Interpreta indicadores sin adjuntar la tabla fuente.","salida_resumida":"La respuesta solicita los datos faltantes.","tiempo_skill_s":5.5,"tiempo_base_s":240.0,"acierto":0,"observaciones":"El criterio de esta prueba exigía analizar datos adjuntos."},
    {"id_caso":"C06","caso_uso":"Educación","consulta":"Adapta una explicación técnica para principiantes.","salida_resumida":"Explicación simplificada con ejemplo.","tiempo_skill_s":9.1,"tiempo_base_s":360.0,"acierto":1,"observaciones":"Evaluar claridad con el público objetivo."}
]
df = pd.DataFrame(datos)
df


## 3. Validación y limpieza básica

Se comprueba que existan las columnas requeridas, que los tiempos sean positivos y que `acierto` solo contenga valores binarios. En una evaluación real, registre también errores, reintentos, versión del modelo y configuración de la Skill.


In [ ]:
columnas_requeridas = {
    "id_caso", "caso_uso", "consulta", "salida_resumida",
    "tiempo_skill_s", "tiempo_base_s", "acierto", "observaciones"
}
faltantes = columnas_requeridas - set(df.columns)
assert not faltantes, f"Faltan columnas: {faltantes}"
assert (df["tiempo_skill_s"] > 0).all(), "Hay tiempos de Skill no positivos."
assert (df["tiempo_base_s"] > 0).all(), "Hay tiempos de referencia no positivos."
assert set(df["acierto"].dropna().unique()).issubset({0, 1}), "acierto debe ser 0 o 1."
print(f"Casos registrados: {len(df)}")
print("Validaciones estructurales completadas.")


## 4. Cálculo de métricas

- **Tiempo medio y mediano:** duración de las ejecuciones.
- **Tasa de acierto:** porcentaje de casos que cumplen los criterios.
- **Reducción estimada de tiempo:** comparación con el proceso de referencia.

La reducción calculada es ilustrativa y depende de que los tiempos base sean comparables y estén medidos de forma consistente.


In [ ]:
tiempo_medio = df["tiempo_skill_s"].mean()
tiempo_mediano = df["tiempo_skill_s"].median()
tasa_acierto = df["acierto"].mean() * 100
tiempo_base_medio = df["tiempo_base_s"].mean()
reduccion_estimada_pct = (
    (df["tiempo_base_s"].sum() - df["tiempo_skill_s"].sum())
    / df["tiempo_base_s"].sum()
) * 100

metricas = pd.DataFrame({
    "Métrica": [
        "Tiempo medio de respuesta (s)",
        "Mediana del tiempo de respuesta (s)",
        "Tasa de acierto (%)",
        "Tiempo medio de referencia (s)",
        "Reducción estimada de tiempo (%)"
    ],
    "Valor": [
        round(tiempo_medio, 2), round(tiempo_mediano, 2),
        round(tasa_acierto, 2), round(tiempo_base_medio, 2),
        round(reduccion_estimada_pct, 2)
    ]
})
metricas


## 5. Comparación por caso de uso

La segmentación permite observar variaciones entre áreas. Debido al tamaño reducido y simulado de la muestra, los resultados son demostrativos y no deben generalizarse.


In [ ]:
resumen_area = (
    df.groupby("caso_uso", as_index=False)
      .agg(
          casos=("id_caso", "count"),
          tiempo_medio_s=("tiempo_skill_s", "mean"),
          tasa_acierto_pct=("acierto", lambda s: s.mean() * 100)
      )
      .round(2)
)
resumen_area


In [ ]:
ax = resumen_area.plot(
    x="caso_uso", y="tiempo_medio_s", kind="bar",
    legend=False, figsize=(9, 4), rot=20,
    title="Tiempo medio de respuesta por caso de uso"
)
ax.set_xlabel("Caso de uso")
ax.set_ylabel("Tiempo medio (segundos)")
plt.tight_layout()
plt.show()


## 6. Exportación de resultados

Los resultados se exportan a CSV para incorporarlos a un informe o revisarlos en una hoja de cálculo.


In [ ]:
df.to_csv("datos_prueba_skills_gemini.csv", index=False, encoding="utf-8-sig")
metricas.to_csv("metricas_skills_gemini.csv", index=False, encoding="utf-8-sig")
print("Exportados: datos_prueba_skills_gemini.csv y metricas_skills_gemini.csv")


## 7. Conclusiones de la validación

1. **Potencial de eficiencia:** las Skills pueden reducir trabajo repetitivo cuando las tareas y los criterios están bien definidos.
2. **Consistencia:** las instrucciones reutilizables ayudan a mantener estructuras y procedimientos comunes.
3. **Evaluación integral:** la rapidez no garantiza exactitud; deben medirse calidad, cumplimiento de formato, utilidad y seguridad.
4. **Límite de la evidencia:** los datos de este cuaderno son simulados y no constituyen evidencia del rendimiento real de Gemini.
5. **Próximo paso:** ejecutar casos representativos, repetir cada prueba, comparar con una línea base y documentar errores, latencia, configuración y evaluación de expertos.

### Recomendaciones para una validación real

- Definir previamente una rúbrica de evaluación.
- Usar consultas representativas y datos autorizados o anonimizados.
- Repetir cada caso y reportar media, mediana y variabilidad.
- Registrar fecha, modelo, configuración y versión de la Skill.
- No introducir información confidencial sin autorización institucional.

---
**Fin del cuaderno.** Plantilla técnica reproducible para fines académicos; sustituya los datos de ejemplo por resultados observados antes de presentar conclusiones empíricas.
